# Valley Bottom Extraction (VBET)
## Maximum Riparian Corridor Extent — Cheyenne River, SD

This notebook delineates the valley bottom (maximum riparian corridor extent) for the
Cheyenne River corridor, South Dakota. The output polygon serves as the analysis extent
for cottonwood gallery classification from multispectral imagery in later notebooks.

> **Run `01a_DEM_Prefetch.ipynb` first.** This notebook reads a local DEM mosaic and a
> drainage-area-annotated flowline layer that notebook 01a produces. It no longer downloads
> anything large.

### Methodology
Adapted from the [VBET-2](https://github.com/jtgilbert/VBET-2) open-source tool and
Woodward et al. (2018) CO-RIP methodology. Core metric is
**HAND (Height Above Nearest Drainage)** — the vertical distance from each terrain cell
to its nearest stream cell along the flow path. Valley bottoms have low HAND values;
hillslopes have high values.

### Pipeline
1. AOI + NHD flowlines (with drainage area) — from notebook 00 / 01a
2. Local DEM mosaic — from notebook 01a (USGS 3DEP static tiles)
3. Hydrological conditioning (**WhiteboxTools**)
4. HAND computation (**WhiteboxTools** `ElevationAboveStream`)
5. Valley bottom delineation (drainage-area-aware thresholds)
6. Post-processing + export
7. Visualization & validation

### Scaling notes
The corridor is ~6,225 km² over a 228 × 187 km envelope — about **47 million cells at 30 m**.
Two changes make that tractable:

- **WhiteboxTools instead of pysheds** for the hydrology. WBT is a multithreaded Rust engine
  that streams rasters to and from disk, so it does not hold several full-grid `float32` arrays
  in Python memory simultaneously. `BreachDepressionsLeastCost` also replaces the
  `fill_pits → fill_depressions → resolve_flats` chain, which was the slowest step.
- **A local DEM** instead of a live `py3dep` request. See notebook 01a for why.

If you still run out of memory, flip `USE_HUC_CHUNKING = True` in the configuration cell —
see Section 8.

### References
- Woodward, B.D. et al. (2018). CO-RIP: A Riparian Vegetation and Corridor Extent Dataset. *ISPRS Int. J. Geo-Inf.*, 7(10), 397. https://doi.org/10.3390/ijgi7100397
- Gilbert, J.T. VBET-2. https://github.com/jtgilbert/VBET-2
- Nobre et al. (2011). Height Above the Nearest Drainage. *J. Hydrology*, 404(1–2), 13–29.
- Lindsay, J.B. (2016). WhiteboxTools. https://www.whiteboxgeo.com/

## 0. Imports & Setup

In [ ]:
import os, sys

# --- PROJ/GDAL data paths: must be set BEFORE any geospatial import ---
# The Jupyter kernel starts without `conda activate`, so these are otherwise unset.
# sys.prefix alone is not enough: in a venv layered on a conda env (the CyVerse
# HYR-SENSE overlay), sys.prefix is the venv, which has no share/proj — the data
# lives under sys.base_prefix. Check both and use whichever actually exists.
def _find_share(name):
    for base in (sys.prefix, sys.base_prefix):
        p = os.path.join(base, "share", name)
        if os.path.isdir(p):
            return p
    return None

_proj, _gdal = _find_share("proj"), _find_share("gdal")
if _proj:
    os.environ["PROJ_DATA"] = os.environ["PROJ_LIB"] = _proj
if _gdal:
    os.environ.setdefault("GDAL_DATA", _gdal)

import warnings
warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=UserWarning)

import time
from pathlib import Path

import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt

import rasterio
from rasterio.features import shapes, rasterize
import rioxarray as rxr

from shapely.geometry import box, shape
from shapely.ops import unary_union

import folium
from scipy.ndimage import label, binary_fill_holes

# WhiteboxTools does all the hydrology (see 01a for install / persistence notes)
import whitebox

# Data directory — override with VBET_DATA_DIR (point at ~/data-store/ on CyVerse)
DATA_DIR = Path(os.environ.get("VBET_DATA_DIR", "../data")).expanduser()
DATA_DIR.mkdir(parents=True, exist_ok=True)

print("Imports OK")
print(f"  DATA_DIR: {DATA_DIR.resolve()}")

---
## 1. Configuration

**Edit this cell to control the run.**

Inputs come from notebook 01a. `BUFFER_KM` and `DEM_RESOLUTION` **must match** what 01a used,
because the DEM mosaic on disk was already clipped and resampled to those settings — changing
them here does not re-cut the DEM.

### Stream initiation threshold
`FLOW_ACCUM_THRESHOLD` is now **derived from resolution** rather than hardcoded. The old value
(`500` cells) was commented as "~50 km² at 10 m", but at 30 m the same 500 cells is 450 km² —
which would strip out every tributary and drive HAND up across the whole corridor. Setting a
contributing-area target in km² keeps the derived network stable if you change resolution.


In [ ]:
# ============================================================
# CONFIGURATION
# ============================================================

# ---- Inputs (produced by 00_Study_Area-Cottonwoods.ipynb and 01a_DEM_Prefetch.ipynb) ----
AOI_CORRIDOR_GPKG = DATA_DIR / "cheyenne_corridor_aoi.gpkg"      # notebook 00
FLOWLINES_GPKG    = DATA_DIR / "cheyenne_flowlines_vaa.gpkg"     # notebook 01a (has totdasqkm)

# ---- Must match 01a ----
DEM_RESOLUTION = 30          # m
BUFFER_KM      = 5           # buffer applied to the AOI when the DEM was cut
CRS_PROJ       = "EPSG:32613"  # UTM Zone 13N

# ---- Stream initiation: contributing area at which a channel begins ----
# 5 km² is a reasonable semi-arid value; lower it to densify the derived network.
STREAM_INIT_KM2      = 5.0
FLOW_ACCUM_THRESHOLD = int(STREAM_INIT_KM2 * 1e6 / DEM_RESOLUTION**2)

# ---- Depression breaching search distance (cells) ----
# Least-cost breaching carves through blockages up to this far; larger is more
# thorough but slower. 100 cells = 3 km at 30 m.
BREACH_DIST_CELLS = 100

# ---- VBET thresholds by drainage area class ----
# Each entry: drainage area window (km²), max HAND (m), max slope (deg), search buffer (m)
VBET_CLASSES = [
    {"label": "large",  "da_min": 1000,  "da_max": 1e9,  "hand_m": 12, "slope_deg": 6,  "buffer_m": 1000},
    {"label": "medium", "da_min": 100,   "da_max": 1000, "hand_m": 8,  "slope_deg": 8,  "buffer_m": 500},
    {"label": "small",  "da_min": 0,     "da_max": 100,  "hand_m": 4,  "slope_deg": 12, "buffer_m": 200},
]

# Minimum valley bottom patch size to keep (hectares)
MIN_PATCH_HA = 1.0

# Set True only if the single-pass run exhausts memory — see Section 8.
USE_HUC_CHUNKING = False

# ---- Outputs ----
OUTPUT_GPKG = DATA_DIR / "cheyenne_valley_bottom.gpkg"

print("Configuration set")
print(f"  DEM resolution       : {DEM_RESOLUTION} m")
print(f"  Stream initiation    : {STREAM_INIT_KM2} km² "
      f"= {FLOW_ACCUM_THRESHOLD:,} cells")
print(f"  Breach search dist   : {BREACH_DIST_CELLS} cells "
      f"({BREACH_DIST_CELLS * DEM_RESOLUTION / 1000:.1f} km)")
print(f"  HUC chunking         : {USE_HUC_CHUNKING}")

---
## 2. AOI & Flowlines

Load the corridor polygon from notebook 00 and the drainage-area-annotated flowlines from
notebook 01a. The flowlines drive the VBET classing: each reach is assigned to a size class by
its NHDPlus total drainage area (`totdasqkm`), and each class gets its own HAND, slope, and
search-buffer thresholds.

In [ ]:
if not AOI_CORRIDOR_GPKG.exists():
    raise FileNotFoundError(
        f"{AOI_CORRIDOR_GPKG} not found — run 00_Study_Area-Cottonwoods.ipynb first."
    )

# ---- Corridor AOI ----
aoi_gdf  = gpd.read_file(AOI_CORRIDOR_GPKG, layer="study_area").to_crs(4326)
aoi_geom = aoi_gdf.geometry.iloc[0]
aoi_proj = aoi_gdf.to_crs(CRS_PROJ)

# ---- Flowlines: prefer the VAA-joined layer from 01a ----
if FLOWLINES_GPKG.exists():
    flowlines = gpd.read_file(FLOWLINES_GPKG, layer="flowlines").to_crs(4326)
    print(f"Flowlines from {FLOWLINES_GPKG.name} (drainage area attached)")
else:
    flowlines = gpd.read_file(AOI_CORRIDOR_GPKG, layer="flowlines").to_crs(4326)
    print(f"WARNING: {FLOWLINES_GPKG.name} not found — falling back to the corridor gpkg,\n"
          f"         which has no drainage area. Run 01a to enable VBET size classing.")

# ---- Buffered footprint (matches the DEM cut in 01a) ----
dem_aoi_proj = aoi_proj.copy()
dem_aoi_proj["geometry"] = aoi_proj.geometry.buffer(BUFFER_KM * 1000)
dem_aoi_wgs84 = dem_aoi_proj.to_crs(4326)

flowlines_proj = flowlines.to_crs(CRS_PROJ).clip(dem_aoi_proj)

print(f"\nStudy area      : {aoi_proj.area.sum() / 1e6:,.0f} km²")
print(f"Buffered ({BUFFER_KM} km): {dem_aoi_proj.area.sum() / 1e6:,.0f} km²")
print(f"Flowlines       : {len(flowlines_proj):,} reaches, "
      f"{flowlines_proj.geometry.length.sum() / 1000:,.0f} km")
print(f"AOI bounds (WGS84): {dem_aoi_wgs84.total_bounds}")

In [ ]:
# ---- Assign each reach to a VBET size class ----
# Previously this happened inline in the delineation cell and silently degraded: the
# corridor gpkg carries only 'nhdplus_comid', so the lookup for 'totdasqkm' failed and
# EVERY reach fell into the 'medium' class. The Cheyenne main stem (>3,000 km²) was
# therefore delineated with headwater thresholds. Doing it here makes the outcome visible.

DA_COL = next((c for c in ["totdasqkm", "totDASqKm", "drainage_area_km2"]
               if c in flowlines_proj.columns), None)

if DA_COL is None:
    print("No drainage area column — all reaches assigned to 'medium'.")
    print("Run 01a_DEM_Prefetch.ipynb to attach NHDPlus VAA and enable size classing.")
    flowlines_proj["vbet_class"] = "medium"
else:
    da = pd.to_numeric(flowlines_proj[DA_COL], errors="coerce")

    # Reaches with no VAA match fall back to stream order, which correlates well
    # enough with drainage area to keep them out of the wrong class entirely.
    if da.isna().any() and "streamorde" in flowlines_proj.columns:
        so = pd.to_numeric(flowlines_proj["streamorde"], errors="coerce")
        da = da.fillna(so.map(lambda s: 2000.0 if s >= 6 else (300.0 if s >= 4 else 10.0)))
        print(f"Filled {int(pd.to_numeric(flowlines_proj[DA_COL], errors='coerce').isna().sum()):,} "
              f"missing drainage areas from stream order")

    flowlines_proj["_da"] = da
    conditions = [(da >= c["da_min"]) & (da < c["da_max"]) for c in VBET_CLASSES]
    flowlines_proj["vbet_class"] = np.select(
        conditions, [c["label"] for c in VBET_CLASSES], default="small"
    )

print("\nReaches per VBET class:")
for cls in VBET_CLASSES:
    sel = flowlines_proj[flowlines_proj["vbet_class"] == cls["label"]]
    print(f"  {cls['label']:6s} (HAND<{cls['hand_m']:2d}m, slope<{cls['slope_deg']:2d}°, "
          f"buf {cls['buffer_m']:4d}m): {len(sel):6,} reaches, "
          f"{sel.geometry.length.sum() / 1000:8,.0f} km")

---
## 3. Local DEM

The DEM is **read from disk, not downloaded**. Notebook 01a builds it from static USGS 3DEP
COG tiles pulled from S3, mosaicked and warped to UTM 13N in a single GDAL pass.

The previous approach — one `py3dep.get_dem()` call over the whole corridor — hit the 3DEP
*dynamic* service, which renders elevation on demand. That is fine for a few hundred km² and
unworkable for 6,225 km². Switching to the pre-staged tiles turns a multi-hour, failure-prone
request into a ~400 MB one-time download.

In [ ]:
dem_path = DATA_DIR / f"cheyenne_dem_{DEM_RESOLUTION}m.tif"

if not dem_path.exists():
    raise FileNotFoundError(
        f"{dem_path} not found.\n\n"
        f"Run 01a_DEM_Prefetch.ipynb first — it downloads the USGS 3DEP tiles and builds\n"
        f"this mosaic. If you set VBET_DATA_DIR, make sure it is set here too\n"
        f"(currently: {DATA_DIR.resolve()})."
    )

with rasterio.open(dem_path) as src:
    dem_shape     = (src.height, src.width)
    dem_transform = src.transform
    dem_crs       = src.crs
    res           = abs(src.transform[0])
    dem_nodata    = src.nodata

n_cells = dem_shape[0] * dem_shape[1]
print(f"DEM: {dem_path.name}")
print(f"  Size      : {dem_shape[1]:,} x {dem_shape[0]:,} ({n_cells / 1e6:.0f} M cells)")
print(f"  Resolution: {res:.1f} m")
print(f"  CRS       : {dem_crs}")
print(f"  NoData    : {dem_nodata}")

if str(dem_crs) != CRS_PROJ:
    print(f"\n  WARNING: DEM CRS is {dem_crs}, config expects {CRS_PROJ}.")
if abs(res - DEM_RESOLUTION) > 0.5:
    print(f"\n  WARNING: DEM is {res:.1f} m but DEM_RESOLUTION is {DEM_RESOLUTION}. "
          f"Thresholds derived from resolution will be wrong.")

# Rough memory estimate for the masking step (HAND + slope + labels held together)
print(f"\n  Peak in-memory arrays for masking: ~{n_cells * 13 / 1e9:.1f} GB")
print(f"  If that exceeds available RAM, set USE_HUC_CHUNKING = True (Section 8).")

In [ ]:
# ---- Overview preview ----
# Read a decimated version rather than the full 47M-cell grid: this is a sanity check,
# not an analysis, and a full read here would waste ~200 MB for nothing.
PREVIEW_MAX_PX = 1500

def read_preview(path, max_px=PREVIEW_MAX_PX, masked=True):
    """Decimated read of a raster plus its plotting extent (left, right, bottom, top)."""
    with rasterio.open(path) as src:
        f = max(1, int(max(src.height, src.width) / max_px))
        out_shape = (src.height // f, src.width // f)
        arr = src.read(1, out_shape=out_shape, masked=masked).astype(np.float32).filled(np.nan) \
            if masked else src.read(1, out_shape=out_shape).astype(np.float32)
        b = src.bounds
    return arr, (b.left, b.right, b.bottom, b.top)

dem_prev, extent = read_preview(dem_path)

fig, ax = plt.subplots(figsize=(13, 7))
im = ax.imshow(dem_prev, cmap="terrain", extent=extent, origin="upper")
flowlines_proj.plot(ax=ax, color="#1565C0", linewidth=0.3)
plt.colorbar(im, ax=ax, label="Elevation (m)", shrink=0.7)
ax.set_title(f"DEM mosaic ({DEM_RESOLUTION} m) with NHD flowlines")
ax.set_aspect("equal")
plt.tight_layout()
plt.show()

---
## 4. Hydrological Conditioning & HAND (WhiteboxTools)

Everything in this section is file-to-file: WhiteboxTools reads and writes GeoTIFFs directly
and nothing full-grid is held in Python memory.

| Step | Tool | Purpose |
|---|---|---|
| 1 | `BreachDepressionsLeastCost` | Carve outlets through depressions so water can route |
| 2 | `D8Pointer` | Flow direction — each cell drains to its steepest neighbour |
| 3 | `D8FlowAccumulation` | Count of upstream contributing cells |
| 4 | `ExtractStreams` | Threshold accumulation into a stream network |
| 5 | `ElevationAboveStream` | **HAND** — elevation above the nearest stream *along the flow path* |
| 6 | `Slope` | Slope in degrees |

### Why breaching rather than filling
The previous pysheds chain was `fill_pits → fill_depressions → resolve_flats`. Filling raises
terrain until depressions overflow, which in flat semi-arid basins creates large artificial flat
areas that then need `resolve_flats` to route across. Least-cost **breaching** instead cuts a
minimal channel through the blockage, preserving the original surface almost everywhere. It is
both more faithful for HAND and substantially faster.

`ElevationAboveStream` measures HAND along the D8 flow path, which is the definition in
Nobre et al. (2011). WhiteboxTools also ships `ElevationAboveStreamEuclidean`, which uses
straight-line distance instead — faster, but it will jump across drainage divides in a
meandering corridor like this one, so it is not used here.

In [ ]:
# ---- Initialise WhiteboxTools ----
# WBT_PATH must be set before WhiteboxTools() is constructed: the constructor calls
# download_wbt(), which returns early when that variable is set. Without it, CyVerse
# re-downloads the ~200 MB binary every session because /opt/conda does not persist.
WBT_PERSIST_DIR = Path.home() / "data-store" / "bin" / "WBT"
WBT_EXE = "whitebox_tools.exe" if sys.platform.startswith("win") else "whitebox_tools"

if (WBT_PERSIST_DIR / WBT_EXE).exists():
    os.environ["WBT_PATH"] = str(WBT_PERSIST_DIR)
    print(f"Using persistent WhiteboxTools at {WBT_PERSIST_DIR}")

wbt = whitebox.WhiteboxTools()
wbt.verbose = False
if (WBT_PERSIST_DIR / WBT_EXE).exists():
    wbt.set_whitebox_dir(str(WBT_PERSIST_DIR))

wbt.set_max_procs(-1)                          # -1 = all cores
wbt.set_working_dir(str(DATA_DIR.resolve()))   # WBT resolves bare filenames against this

WBT_VERSION = wbt.version().splitlines()[0].strip()
print(f"{WBT_VERSION}")
print(f"  exe dir: {wbt.exe_path}")
print(f"  workdir: {DATA_DIR.resolve()}")

In [ ]:
# ============================================================
# HYDROLOGY PIPELINE — file-to-file, nothing full-grid in Python memory
# ============================================================
R = DEM_RESOLUTION
breached_path = DATA_DIR / f"cheyenne_dem_{R}m_breached.tif"
fdir_path     = DATA_DIR / f"cheyenne_fdir_{R}m.tif"
facc_path     = DATA_DIR / f"cheyenne_facc_{R}m.tif"
streams_path  = DATA_DIR / f"cheyenne_streams_{R}m.tif"
hand_path     = DATA_DIR / f"cheyenne_hand_{R}m.tif"
slope_path    = DATA_DIR / f"cheyenne_slope_{R}m.tif"

# Bare filenames: WBT resolves them against the working directory set above.
n = lambda p: p.name

def run_step(name, out_path, fn):
    """Run one WBT step unless its output is already cached. Returns elapsed seconds."""
    if out_path.exists():
        print(f"  [cached] {name:28s} {out_path.name}")
        return 0.0
    t0 = time.time()
    print(f"  [run   ] {name:28s} …", end="", flush=True)
    rc = fn()
    dt = time.time() - t0
    if rc != 0 or not out_path.exists():
        raise RuntimeError(f"WhiteboxTools step '{name}' failed (exit {rc}). "
                           f"Re-run with wbt.verbose = True to see the tool output.")
    print(f" done in {dt / 60:.1f} min")
    return dt

print(f"Hydrology pipeline on {n_cells / 1e6:.0f} M cells\n"
      f"(delete an output file to force that step to re-run)\n")
t_start = time.time()

run_step("BreachDepressionsLeastCost", breached_path, lambda: wbt.breach_depressions_least_cost(
    n(dem_path), n(breached_path), dist=BREACH_DIST_CELLS, fill=True))

run_step("D8Pointer", fdir_path, lambda: wbt.d8_pointer(
    n(breached_path), n(fdir_path)))

run_step("D8FlowAccumulation", facc_path, lambda: wbt.d8_flow_accumulation(
    n(fdir_path), n(facc_path), out_type="cells", pntr=True))

run_step("ExtractStreams", streams_path, lambda: wbt.extract_streams(
    n(facc_path), n(streams_path), threshold=FLOW_ACCUM_THRESHOLD))

run_step("ElevationAboveStream (HAND)", hand_path, lambda: wbt.elevation_above_stream(
    n(breached_path), n(streams_path), n(hand_path)))

run_step("Slope", slope_path, lambda: wbt.slope(
    n(breached_path), n(slope_path), units="degrees"))

print(f"\nHydrology complete in {(time.time() - t_start) / 60:.1f} min total")

In [ ]:
# ---- Verify the derived network against NHD ----
# The derived streams should track the NHD flowlines on the main stem. A systematic offset
# means the breach settings or the threshold are wrong; a *slight* wander is expected,
# since NHD is surveyed hydrography and this is terrain-derived.
facc_prev, extent = read_preview(facc_path)
strm_prev, _      = read_preview(streams_path)

fig, axes = plt.subplots(1, 2, figsize=(17, 6))

im = axes[0].imshow(np.log1p(np.nan_to_num(facc_prev, nan=0)), cmap="Blues",
                    extent=extent, origin="upper")
axes[0].set_title("Flow accumulation (log scale)")
plt.colorbar(im, ax=axes[0], label="log(1 + cells)", shrink=0.7)

axes[1].imshow(np.where(np.nan_to_num(strm_prev, nan=0) > 0, 1, np.nan),
               cmap="autumn", extent=extent, origin="upper")
flowlines_proj.plot(ax=axes[1], color="#1565C0", linewidth=0.4)
axes[1].set_title(f"Derived streams (red, ≥ {STREAM_INIT_KM2} km²)\nvs NHD flowlines (blue)")

for ax in axes:
    ax.set_aspect("equal")
    ax.set_xticks([]); ax.set_yticks([])
plt.tight_layout()
plt.show()

with rasterio.open(streams_path) as src:
    n_stream = int(sum(np.count_nonzero(src.read(1, window=w) > 0)
                       for _, w in src.block_windows(1)))
print(f"Derived stream cells: {n_stream:,} "
      f"({n_stream * res / 1000:,.0f} km of channel)")
print(f"NHD flowlines in AOI: {flowlines_proj.geometry.length.sum() / 1000:,.0f} km")

---
## 5. HAND — inspection

HAND was computed in the pipeline above. It measures the vertical distance from each terrain
cell to the elevation of its nearest stream cell, traced along the flow path:

`HAND[cell] = elevation[cell] − elevation[nearest_stream_cell_along_flow_path]`

Cells in the floodplain have low HAND; hillslopes and uplands have high values. This is the
single metric the valley bottom delineation keys on.

Reference: Nobre et al. (2011), *J. Hydrology*, 404(1–2).

In [ ]:
# ---- HAND distribution (sampled, not a full read) ----
with rasterio.open(hand_path) as src:
    hand_prev = src.read(
        1, out_shape=(src.height // 8, src.width // 8), masked=True
    ).astype(np.float32).filled(np.nan)

finite = hand_prev[np.isfinite(hand_prev)]
pcts = np.percentile(finite, [50, 75, 90, 95, 99])
print("HAND distribution (8x decimated sample):")
for p, v in zip([50, 75, 90, 95, 99], pcts):
    print(f"  p{p:<3d}: {v:7.1f} m")

# What fraction of the AOI each class threshold would admit, before slope/buffer masking
print("\nShare of cells below each class HAND threshold:")
for cls in VBET_CLASSES:
    frac = (finite < cls["hand_m"]).mean()
    print(f"  {cls['label']:6s} (< {cls['hand_m']:2d} m): {frac * 100:5.1f}%")

fig, ax = plt.subplots(figsize=(9, 3.5))
ax.hist(finite[finite < np.percentile(finite, 98)], bins=120, color="#455A64")
for cls in VBET_CLASSES:
    ax.axvline(cls["hand_m"], ls="--", lw=1.2,
               label=f"{cls['label']} ({cls['hand_m']} m)")
ax.set_xlabel("HAND (m)"); ax.set_ylabel("cells")
ax.set_title("HAND distribution with VBET class thresholds")
ax.legend(fontsize=8)
plt.tight_layout()
plt.show()

In [ ]:
# ---- HAND map ----
hand_map, extent = read_preview(hand_path)
vmax = float(np.nanpercentile(hand_map, 95))

fig, ax = plt.subplots(figsize=(14, 7))
im = ax.imshow(hand_map, cmap="RdYlGn_r", vmin=0, vmax=vmax,
               extent=extent, origin="upper")
flowlines_proj.plot(ax=ax, color="#1565C0", linewidth=0.3)
plt.colorbar(im, ax=ax, label="HAND (m above nearest drainage)", shrink=0.7)
ax.set_title("Height Above Nearest Drainage (HAND)")
ax.set_aspect("equal")
plt.tight_layout()
plt.show()

---
## 6. Valley Bottom Delineation — Simplified VBET-2 Logic

Apply drainage-area-aware HAND and slope thresholds following VBET-2.

**Rationale**: Large rivers occupy broad, flat floodplains, so a higher HAND threshold is
appropriate; small headwater streams are confined to narrow steep valleys and need a lower one.
Each NHD reach is assigned a drainage area class (done in Section 2), buffered by that class's
search distance, and only pixels inside the buffer that satisfy *both* the HAND and slope
criteria are kept.

| Class | Drainage Area | Max HAND | Max Slope | Search Buffer |
|-------|--------------|----------|-----------|---------------|
| Large | > 1000 km² | 12 m | 6° | 1000 m |
| Medium | 100–1000 km² | 8 m | 8° | 500 m |
| Small | < 100 km² | 4 m | 12° | 200 m |

> These thresholds now actually apply per class. Before the VAA join in notebook 01a, the
> drainage area lookup failed silently and every reach used the `medium` row.

In [ ]:
# ---- Load HAND and slope as full arrays ----
# These are the only two full-grid arrays we need in memory. Slope now comes from
# WhiteboxTools rather than np.gradient, which mishandled the raster edges and ignored
# nodata (producing enormous spurious slopes along the AOI cutline).

print("Loading HAND and slope…")
with rasterio.open(hand_path) as src:
    hand_arr = src.read(1, masked=True).astype(np.float32).filled(np.nan)
with rasterio.open(slope_path) as src:
    slope_arr = src.read(1, masked=True).astype(np.float32).filled(np.nan)

assert hand_arr.shape == dem_shape, f"HAND shape {hand_arr.shape} != DEM {dem_shape}"
assert slope_arr.shape == dem_shape, f"Slope shape {slope_arr.shape} != DEM {dem_shape}"

# HAND is a height above a datum, so negatives are numerical noise, not data
hand_arr = np.where(hand_arr < 0, np.nan, hand_arr)

valid = np.isfinite(hand_arr) & np.isfinite(slope_arr)
print(f"  Valid cells : {valid.sum():,} / {hand_arr.size:,} "
      f"({100 * valid.sum() / hand_arr.size:.1f}%)")
print(f"  HAND  p99   : {np.nanpercentile(hand_arr, 99):.1f} m")
print(f"  Slope p99   : {np.nanpercentile(slope_arr, 99):.1f}°")
print(f"  Memory held : {(hand_arr.nbytes + slope_arr.nbytes) / 1e9:.2f} GB")

In [ ]:
# ============================================================
# VBET DELINEATION
# ============================================================
valley_mask = np.zeros(dem_shape, dtype=np.uint8)
t0 = time.time()

for cls in VBET_CLASSES:
    class_flw = flowlines_proj[flowlines_proj["vbet_class"] == cls["label"]]
    if len(class_flw) == 0:
        print(f"  {cls['label']:6s}: no reaches, skipping")
        continue

    # Rasterize the buffered reaches directly.
    #
    # This previously did `.buffer(d).union_all()` first, then rasterized one giant
    # multipolygon. Unioning thousands of overlapping buffers is by far the most expensive
    # geometry operation in the notebook — and it buys nothing, because rasterizing
    # overlapping polygons to the same burn value already *is* a union.
    buffer_raster = rasterize(
        ((geom, 1) for geom in class_flw.geometry.buffer(cls["buffer_m"])),
        out_shape=dem_shape,
        transform=dem_transform,
        fill=0,
        dtype=np.uint8,
        all_touched=True,
    )

    class_mask = (
        (buffer_raster == 1)
        & (hand_arr < cls["hand_m"])
        & (slope_arr < cls["slope_deg"])
        & valid
    )

    np.maximum(valley_mask, class_mask, out=valley_mask, casting="unsafe")
    print(f"  {cls['label']:6s}: {len(class_flw):6,} reaches -> "
          f"{int(class_mask.sum()):10,} cells "
          f"({int(class_mask.sum()) * res**2 / 1e6:7.1f} km²)")
    del buffer_raster, class_mask

print(f"\nCombined valley mask: {int(valley_mask.sum()):,} cells "
      f"({int(valley_mask.sum()) * res**2 / 1e6:.1f} km²) in {time.time() - t0:.0f} s")
print(f"  = {100 * valley_mask.sum() * res**2 / 1e6 / (aoi_proj.area.sum() / 1e6):.1f}% "
      f"of the study area")

---
## 6. Post-Processing & Export

1. **Fill holes** — fill enclosed upland patches inside the valley bottom
2. **Remove slivers** — drop isolated patches smaller than `MIN_PATCH_HA`
3. **Smooth boundaries** — buffer then un-buffer to remove jagged edges
4. **Vectorize** — convert raster mask to polygon
5. **Export** — save as GeoPackage

In [ ]:
# ---- Morphological cleanup ----
print("Post-processing valley mask…")
t0 = time.time()

# Fill enclosed upland patches inside the valley bottom
valley_filled = binary_fill_holes(valley_mask).astype(np.uint8)
del valley_mask

# Drop patches below the minimum area.
#
# This was a per-label loop: `for i in 1..n: if (labeled == i).sum() < min_cells`, which
# scans all 47M cells once per component. With thousands of components that dominated the
# whole notebook's runtime. np.bincount counts every label in a single pass instead.
labeled, n_features = label(valley_filled)
min_cells = int((MIN_PATCH_HA * 1e4) / (res**2))

counts = np.bincount(labeled.ravel())
counts[0] = 0                                   # label 0 is background
keep_labels = np.flatnonzero(counts >= min_cells)
valley_filled = np.isin(labeled, keep_labels).astype(np.uint8)
del labeled

print(f"  Components: {n_features:,} -> {len(keep_labels):,} "
      f"(kept those ≥ {MIN_PATCH_HA} ha = {min_cells} cells)")
print(f"  Cells retained: {int(valley_filled.sum()):,} "
      f"({int(valley_filled.sum()) * res**2 / 1e6:.1f} km²)")

# ---- Vectorize ----
# mask=valley_filled restricts the walk to valley cells instead of the whole grid.
print("Vectorizing…")
polygons = [
    shape(geom)
    for geom, val in shapes(valley_filled, mask=valley_filled.astype(bool),
                            transform=dem_transform)
    if val == 1
]
valley_poly = unary_union(polygons)

# Smooth boundaries (buffer out then back in removes pixel-edge jaggedness)
smooth_dist = res * 2
valley_smooth = valley_poly.buffer(smooth_dist).buffer(-smooth_dist)

print(f"  Polygons: {len(polygons):,}")
print(f"  Final valley bottom area: {valley_smooth.area / 1e6:.1f} km² "
      f"({time.time() - t0:.0f} s)")

In [ ]:
# ---- Export to GeoPackage ----
# Provenance: this used to record `gauge_id: GAUGE_ID` even when the AOI came from the
# corridor gpkg, which mislabelled every full-corridor run as a single-gauge run.
valley_gdf = gpd.GeoDataFrame(
    {
        "method":                  ["VBET-simplified"],
        "aoi_source":              [AOI_CORRIDOR_GPKG.name],
        "flowline_source":         [FLOWLINES_GPKG.name if FLOWLINES_GPKG.exists()
                                    else AOI_CORRIDOR_GPKG.name],
        "dem_source":              ["USGS 3DEP 1 arc-second (static COG tiles)"],
        "dem_resolution_m":        [DEM_RESOLUTION],
        "hydrology_engine":        [WBT_VERSION],
        "stream_init_km2":         [STREAM_INIT_KM2],
        "flow_accum_threshold":    [FLOW_ACCUM_THRESHOLD],
        "hand_threshold_m_large":  [VBET_CLASSES[0]["hand_m"]],
        "hand_threshold_m_medium": [VBET_CLASSES[1]["hand_m"]],
        "hand_threshold_m_small":  [VBET_CLASSES[2]["hand_m"]],
        "min_patch_ha":            [MIN_PATCH_HA],
        "area_km2":                [valley_smooth.area / 1e6],
        "created":                 [pd.Timestamp.now().isoformat(timespec="seconds")],
    },
    geometry=[valley_smooth],
    crs=CRS_PROJ,
)

valley_gdf.to_file(OUTPUT_GPKG, layer="valley_bottom", driver="GPKG")
print(f"Valley bottom  -> {OUTPUT_GPKG}")

# Also write the per-reach classing, so downstream notebooks can weight by stream size
flowlines_proj.to_file(OUTPUT_GPKG, layer="flowlines_classed", driver="GPKG")
print(f"Classed reaches -> {OUTPUT_GPKG} (layer 'flowlines_classed')")

# Raster mask for inspection / zonal work
valley_raster_path = DATA_DIR / f"cheyenne_valley_mask_{DEM_RESOLUTION}m.tif"
with rasterio.open(dem_path) as src:
    profile = src.profile.copy()
profile.update(dtype="uint8", count=1, compress="deflate", nodata=0,
               tiled=True, blockxsize=512, blockysize=512)
with rasterio.open(valley_raster_path, "w", **profile) as dst:
    dst.write(valley_filled[np.newaxis, :, :])
print(f"Valley mask    -> {valley_raster_path}")

---
## 7. Visualization & Validation

### 7a. Interactive Folium Map
Overlay the valley bottom polygon on a satellite basemap with NHD flowlines.

In [ ]:
# Reproject to WGS84 for Folium
valley_wgs84 = valley_gdf.to_crs(4326)

# Only the main-stem-scale reaches, simplified: pushing all 8,964 corridor reaches into a
# Folium GeoJson layer produces a many-MB blob that makes the map unusable in the browser.
flw_map = flowlines_proj[flowlines_proj["vbet_class"].isin(["large", "medium"])].copy()
flw_map["geometry"] = flw_map.geometry.simplify(50)   # 50 m tolerance, in UTM metres
flw_map = flw_map[["vbet_class", "geometry"]].to_crs(4326)

bounds = valley_wgs84.total_bounds  # (minx, miny, maxx, maxy)
center = [(bounds[1] + bounds[3]) / 2, (bounds[0] + bounds[2]) / 2]

m = folium.Map(location=center, zoom_start=9, tiles=None)

folium.TileLayer(
    tiles="https://server.arcgisonline.com/ArcGIS/rest/services/World_Imagery/MapServer/tile/{z}/{y}/{x}",
    attr="Esri World Imagery",
    name="Satellite",
    overlay=False,
).add_to(m)

folium.GeoJson(
    valley_wgs84,
    name="Valley Bottom (VBET)",
    style_function=lambda x: {
        "color": "#2ecc71", "weight": 1.5,
        "fillColor": "#2ecc71", "fillOpacity": 0.35,
    },
    tooltip=folium.GeoJsonTooltip(fields=["area_km2"], aliases=["Area (km²)"]),
).add_to(m)

folium.GeoJson(
    flw_map,
    name="NHD Flowlines (large + medium)",
    style_function=lambda f: {
        "color": "#0d47a1" if f["properties"]["vbet_class"] == "large" else "#3498db",
        "weight": 2.5 if f["properties"]["vbet_class"] == "large" else 1.2,
    },
).add_to(m)

folium.LayerControl().add_to(m)
m.fit_bounds([[bounds[1], bounds[0]], [bounds[3], bounds[2]]])
m

### 7b. Cross-Section Profiles

Plot elevation and HAND values along perpendicular transects to visually confirm that the
valley bottom threshold captures the floodplain and stops at the valley walls.

In [ ]:
from shapely.geometry import LineString

def sample_raster_along_line(raster_path, line, n_points=200):
    """Sample a raster along a Shapely LineString, return (distances_km, values)."""
    coords = [line.interpolate(i / n_points, normalized=True) for i in range(n_points + 1)]
    xy = [(c.x, c.y) for c in coords]
    with rasterio.open(raster_path) as src:
        values = np.array([v[0] for v in src.sample(xy)], dtype=np.float32)
        if src.nodata is not None:
            values = np.where(values == src.nodata, np.nan, values)
    dists = np.array([line.project(c) for c in coords]) / 1000
    return dists, values

# Pick the main stem: the longest reach in the largest drainage-area class.
#
# This previously did `sort_values("geometry", key=lambda g: g.length, ...)`, which relies on
# pandas passing the whole GeoSeries to the key function — fragile, and it ignored drainage
# area entirely, so it could land on a long tributary instead of the Cheyenne itself.
candidates = flowlines_proj[flowlines_proj["vbet_class"] == "large"]
if len(candidates) == 0:
    candidates = flowlines_proj
ms_line = candidates.loc[candidates.geometry.length.idxmax()].geometry

print(f"Main stem reach: {ms_line.length / 1000:.1f} km "
      f"(from {len(candidates):,} '{candidates['vbet_class'].iloc[0]}' reaches)")

transect_width_m = VBET_CLASSES[0]["buffer_m"] * 4   # 4 km wide for the large class
fig, axes = plt.subplots(3, 1, figsize=(14, 10))

for ax_i, pct in enumerate([0.25, 0.5, 0.75]):
    pt = ms_line.interpolate(pct, normalized=True)

    # Local flow direction -> perpendicular transect
    delta = 0.01
    pt_ahead = ms_line.interpolate(min(pct + delta, 1.0), normalized=True)
    dx, dy = pt_ahead.x - pt.x, pt_ahead.y - pt.y
    length = np.hypot(dx, dy)
    if length == 0:
        continue
    perp_dx = -dy / length * transect_width_m / 2
    perp_dy = dx / length * transect_width_m / 2

    transect = LineString([(pt.x - perp_dx, pt.y - perp_dy),
                           (pt.x + perp_dx, pt.y + perp_dy)])

    dists, elev       = sample_raster_along_line(str(dem_path), transect)
    dists_h, hand_val = sample_raster_along_line(str(hand_path), transect)
    _, vb             = sample_raster_along_line(str(valley_raster_path), transect)

    ax = axes[ax_i]
    ax2 = ax.twinx()
    color_elev, color_hand = "#795548", "#1565C0"

    # Shade where the delineated valley bottom actually falls
    ax.fill_between(dists, elev.min(), elev.max(), where=(vb == 1),
                    color="#2ecc71", alpha=0.20, step="mid",
                    label="Delineated valley bottom")
    ax.plot(dists, elev, color=color_elev, lw=1.5, label="Elevation (m)")
    ax2.plot(dists_h, hand_val, color=color_hand, lw=1.5, ls="--", label="HAND (m)")
    ax2.axhline(VBET_CLASSES[0]["hand_m"], color=color_hand, lw=0.8, ls=":",
                label=f"HAND threshold (large, {VBET_CLASSES[0]['hand_m']} m)")

    ax.set_title(f"Cross-section at {pct * 100:.0f}% along main stem")
    ax.set_xlabel("Distance along transect (km)")
    ax.set_ylabel("Elevation (m)", color=color_elev)
    ax2.set_ylabel("HAND (m)", color=color_hand)
    ax.tick_params(axis="y", labelcolor=color_elev)
    ax2.tick_params(axis="y", labelcolor=color_hand)

    l1, lb1 = ax.get_legend_handles_labels()
    l2, lb2 = ax2.get_legend_handles_labels()
    ax.legend(l1 + l2, lb1 + lb2, loc="upper right", fontsize=8)

plt.tight_layout()
plt.show()

### 7c. Valley Bottom Area Statistics

In [ ]:
# ---- Summary ----
total_area_km2   = valley_gdf.geometry.area.sum() / 1e6
aoi_area_km2     = aoi_proj.area.sum() / 1e6
total_flowline_km = flowlines_proj.geometry.length.sum() / 1000
avg_width_m      = valley_gdf.geometry.area.sum() / flowlines_proj.geometry.length.sum()

print("=" * 58)
print("VALLEY BOTTOM SUMMARY")
print("=" * 58)
print(f"  Valley bottom area : {total_area_km2:10,.1f} km²")
print(f"  Study area         : {aoi_area_km2:10,.1f} km²  "
      f"({100 * total_area_km2 / aoi_area_km2:.1f}% captured)")
print(f"  Flowline length    : {total_flowline_km:10,.1f} km")
print(f"  Mean valley width  : {avg_width_m:10,.0f} m")
print(f"  DEM resolution     : {DEM_RESOLUTION:10d} m")
print(f"  Stream initiation  : {STREAM_INIT_KM2:10.1f} km²")
print(f"  Hydrology engine   : {WBT_VERSION}")
print(f"  Output             : {OUTPUT_GPKG}")
print("=" * 58)

# Plausibility check — this is a corridor, not a blanket over the AOI
frac = total_area_km2 / aoi_area_km2
if frac > 0.60:
    print("\n  WARNING: the valley bottom covers most of the study area. HAND or slope\n"
          "           thresholds are likely too permissive, or the stream initiation\n"
          "           area is too large (thin network -> inflated HAND).")
elif frac < 0.02:
    print("\n  WARNING: the valley bottom is very small. Check that the derived stream\n"
          "           network tracks NHD (Section 4) before tuning thresholds.")
else:
    print(f"\n  Coverage of {frac * 100:.1f}% is plausible for a river corridor.")

print("\nValley bottom polygon attributes:")
valley_gdf.drop(columns="geometry").T

---
## 8. Optional — HUC-8 chunked processing

**You should not need this.** At 30 m the corridor is ~47 M cells, which WhiteboxTools handles
comfortably in a single pass; the sections above are the intended path. This section exists for
two situations:

1. The CyVerse instance is small and Section 6 runs out of memory loading HAND + slope.
2. You want to re-run at **10 m** (`PRODUCT = "13"` in notebook 01a), where the grid is ~9×
   larger — about 420 M cells — and single-pass is genuinely not viable.

To use it, set `USE_HUC_CHUNKING = True` in the configuration cell and run from here.

### Why HUC-8, and not tiles

This is the part that matters. **Arbitrary rectangular tiles break flow routing.** A tile edge
cuts through hillslopes mid-flowpath, so every chunk loses its upstream contributing area:
flow accumulation collapses near the seam, the derived stream network disappears there, and
HAND — which is measured *to the nearest stream* — becomes meaningless for a wide band around
every boundary. Mosaicking the pieces back together does not repair it.

**HUC boundaries are drainage divides.** By construction, no flow crosses them. Splitting the
DEM by hydrologic unit is therefore very close to lossless for D8 routing: each chunk already
contains the full contributing area for everything inside it. The 5 km buffer below covers the
remaining edge effects (a HUC's own outlet reach, and DEM-derived divides that wander slightly
from the mapped WBD polygon).

The cost is real, though — per-chunk failure handling, seam merging, and roughly 8–12 separate
WBT invocations instead of 6. That is why it is not the default.

### Parallelism note
WhiteboxTools already uses every core on each call. Running many chunks at once *and* letting
each use all cores oversubscribes the CPU and gets slower, not faster. `max_workers` below is
deliberately conservative; raise it only if you also cap `wbt.set_max_procs()` per worker.

In [ ]:
# ============================================================
# HUC-8 CHUNKED VBET  (runs only when USE_HUC_CHUNKING = True)
# ============================================================
if not USE_HUC_CHUNKING:
    print("USE_HUC_CHUNKING is False — skipping. The single-pass result above stands.")
else:
    # ThreadPoolExecutor, not ProcessPoolExecutor: the expensive work happens inside the
    # WhiteboxTools subprocess and in rasterio/numpy, all of which release the GIL. Threads
    # also share flowlines_proj directly and avoid pickling a notebook-local function,
    # which ProcessPoolExecutor cannot do.
    from concurrent.futures import ThreadPoolExecutor, as_completed

    import rasterio.mask
    from rasterio.merge import merge as rio_merge
    from pynhd import WaterData

    CHUNK_DIR = DATA_DIR / f"huc_chunks_{DEM_RESOLUTION}m"
    CHUNK_DIR.mkdir(exist_ok=True)
    HUC_BUFFER_M = 5000     # overlap to absorb divide/outlet edge effects

    # ---- 1. HUC-8 units intersecting the corridor ----
    huc_path = DATA_DIR / "cheyenne_huc8.gpkg"
    if huc_path.exists():
        hucs = gpd.read_file(huc_path)
    else:
        hucs = WaterData("wbd08").bygeom(aoi_geom)
        hucs.to_file(huc_path, driver="GPKG")
    hucs = hucs.to_crs(CRS_PROJ)
    huc_col = next(c for c in ["huc8", "huc_8", "huc8_str"] if c in hucs.columns)
    print(f"{len(hucs)} HUC-8 units intersect the corridor\n")

    def process_huc(huc_id, geom):
        """Full WBT chain for one HUC. Returns the per-HUC valley mask path."""
        work = CHUNK_DIR / huc_id
        work.mkdir(exist_ok=True)
        out_mask = work / "valley_mask.tif"
        if out_mask.exists():
            return out_mask

        # -- Cut the buffered HUC window out of the corridor DEM --
        chunk_dem = work / "dem.tif"
        if not chunk_dem.exists():
            with rasterio.open(dem_path) as src:
                arr, tr = rasterio.mask.mask(src, [geom.buffer(HUC_BUFFER_M)],
                                             crop=True, nodata=-9999.0, filled=True)
                prof = src.profile.copy()
            prof.update(height=arr.shape[1], width=arr.shape[2], transform=tr,
                        nodata=-9999.0, compress="deflate", tiled=True)
            with rasterio.open(chunk_dem, "w", **prof) as dst:
                dst.write(arr)

        # -- WBT chain, scoped to this chunk's directory --
        # Each thread gets its own WhiteboxTools instance: set_working_dir is per-instance
        # state, so a shared one would race.
        w = whitebox.WhiteboxTools()
        w.verbose = False
        if (WBT_PERSIST_DIR / WBT_EXE).exists():
            w.set_whitebox_dir(str(WBT_PERSIST_DIR))
        w.set_working_dir(str(work.resolve()))

        w.breach_depressions_least_cost("dem.tif", "breached.tif",
                                        dist=BREACH_DIST_CELLS, fill=True)
        w.d8_pointer("breached.tif", "fdir.tif")
        w.d8_flow_accumulation("fdir.tif", "facc.tif", out_type="cells", pntr=True)
        w.extract_streams("facc.tif", "streams.tif", threshold=FLOW_ACCUM_THRESHOLD)
        w.elevation_above_stream("breached.tif", "streams.tif", "hand.tif")
        w.slope("breached.tif", "slope.tif", units="degrees")

        # -- Same class thresholds as the single-pass path --
        with rasterio.open(work / "hand.tif") as s:
            h = s.read(1, masked=True).astype(np.float32).filled(np.nan)
            c_shape, c_tr, c_prof = (s.height, s.width), s.transform, s.profile.copy()
        with rasterio.open(work / "slope.tif") as s:
            sl = s.read(1, masked=True).astype(np.float32).filled(np.nan)
        h = np.where(h < 0, np.nan, h)
        ok = np.isfinite(h) & np.isfinite(sl)

        vm = np.zeros(c_shape, dtype=np.uint8)
        local_flw = flowlines_proj[flowlines_proj.intersects(geom.buffer(HUC_BUFFER_M))]
        for cls in VBET_CLASSES:
            sel = local_flw[local_flw["vbet_class"] == cls["label"]]
            if len(sel) == 0:
                continue
            br = rasterize(((g, 1) for g in sel.geometry.buffer(cls["buffer_m"])),
                           out_shape=c_shape, transform=c_tr, fill=0,
                           dtype=np.uint8, all_touched=True)
            np.maximum(vm, ((br == 1) & (h < cls["hand_m"]) &
                            (sl < cls["slope_deg"]) & ok), out=vm, casting="unsafe")

        # -- Clip back to the UNBUFFERED HUC so overlaps don't double-count --
        keep = rasterize([(geom, 1)], out_shape=c_shape, transform=c_tr,
                         fill=0, dtype=np.uint8)
        vm = (vm & keep).astype(np.uint8)

        c_prof.update(dtype="uint8", count=1, nodata=0, compress="deflate", tiled=True)
        with rasterio.open(out_mask, "w", **c_prof) as dst:
            dst.write(vm[np.newaxis, :, :])
        return out_mask

    # ---- 2. Run the chunks ----
    jobs = [(str(r[huc_col]), r.geometry) for _, r in hucs.iterrows()]
    results, failures = [], []
    t0 = time.time()

    # WBT already uses every core per call; running many chunks concurrently AND letting
    # each use all cores oversubscribes the CPU. Keep this small.
    n_workers = max(1, min(4, (os.cpu_count() or 2) // 2))
    print(f"Running {len(jobs)} chunks with {n_workers} worker(s)…")

    with ThreadPoolExecutor(max_workers=n_workers) as ex:
        futures = {ex.submit(process_huc, hid, g): hid for hid, g in jobs}
        for fut in as_completed(futures):
            hid = futures[fut]
            try:
                results.append(fut.result())
                print(f"  HUC {hid}: done ({len(results)}/{len(jobs)})")
            except Exception as exc:
                failures.append((hid, repr(exc)))
                print(f"  HUC {hid}: FAILED — {exc}")

    print(f"\n{len(results)}/{len(jobs)} chunks succeeded in "
          f"{(time.time() - t0) / 60:.1f} min")
    if failures:
        print("Failed chunks (re-run this cell to retry — successes are cached):")
        for hid, err in failures:
            print(f"  {hid}: {err}")
    if not results:
        raise RuntimeError("No chunks succeeded — see the errors above.")

    # ---- 3. Merge ----
    # method="max" so any chunk claiming a cell as valley bottom wins in the overlap.
    srcs = [rasterio.open(p) for p in results]
    try:
        mosaic, out_tr = rio_merge(srcs, method="max")
        prof = srcs[0].profile.copy()
    finally:
        for s in srcs:
            s.close()

    prof.update(height=mosaic.shape[1], width=mosaic.shape[2], transform=out_tr,
                dtype="uint8", count=1, nodata=0, compress="deflate", tiled=True)
    chunked_mask_path = DATA_DIR / f"cheyenne_valley_mask_{DEM_RESOLUTION}m_chunked.tif"
    with rasterio.open(chunked_mask_path, "w", **prof) as dst:
        dst.write(mosaic)

    print(f"\nMerged mask -> {chunked_mask_path}")
    print(f"  {int(mosaic.sum()):,} cells "
          f"({int(mosaic.sum()) * res**2 / 1e6:.1f} km²)")
    print("\nTo vectorize and export this instead of the single-pass mask, set:")
    print("    valley_filled, dem_transform = mosaic[0], out_tr")
    print("and re-run Section 7 (post-processing + export).")

---
## Notes & Next Steps

### Threshold tuning
The HAND and slope thresholds are starting points from the literature for semi-arid rivers.
For the Cheyenne specifically:

- Inspect the cross-sections (Section 7b) — the shaded valley bottom should capture the active
  floodplain and low terraces but stop at the first distinct break in slope.
- Too narrow on the main stem → raise `hand_m` for the `large` class.
- Spilling onto uplands → lower `hand_m` or `slope_deg`.
- Missing tributaries entirely → lower `STREAM_INIT_KM2` so the derived network extends
  further into the headwaters. This matters more than the HAND thresholds do: HAND is measured
  to the *nearest derived stream*, so a sparse network inflates HAND everywhere.
- Compare against the satellite basemap in Section 7a.

Each hydrology output is cached by filename. To re-run after changing `STREAM_INIT_KM2`,
delete `cheyenne_streams_*.tif` and `cheyenne_hand_*.tif` — the expensive breaching and flow
accumulation steps are unaffected by that parameter and will be reused.

### Running on CyVerse
```bash
# Once per session — /opt/conda does not persist
conda config --set fetch_threads 1 && conda config --set extract_threads 1
mamba env create -f environment.yml
conda activate unci-maka-py

# Point all notebook I/O at persistent storage
export VBET_DATA_DIR=/home/jovyan/data-store/unci-maka-data
```
Set the same `VBET_DATA_DIR` in notebooks 00, 01a and 01. Anything written outside
`~/data-store/` is lost when the session ends — including the DEM mosaic and the ~200 MB
WhiteboxTools binary, which is why 01a copies it to `~/data-store/bin/WBT`.

For a long run, `papermill` survives a dropped browser connection where a live kernel does not:
```bash
nohup papermill 01_VBET_ValleyBottom.ipynb 01_VBET_out.ipynb > vbet.log 2>&1 &
```

### Re-running at 10 m
Set `PRODUCT = "13"` and `DEM_RES_M = 10` in notebook 01a, then `DEM_RESOLUTION = 10` here.
The grid grows ~9× to roughly 420 M cells; use Section 8 (`USE_HUC_CHUNKING = True`).
`FLOW_ACCUM_THRESHOLD` rescales automatically from `STREAM_INIT_KM2`.

### Downstream use
`data/cheyenne_valley_bottom.gpkg` is the analysis extent for cottonwood gallery mapping:
- `02_HLS_reflectance.ipynb` — clip HLS imagery to the riparian corridor
- `03_Cottonwood_Training_Data.ipynb` / `04_Image_Classification.ipynb` — restrict training
  samples and classification to within the valley bottom

The `flowlines_classed` layer in the same GeoPackage carries the per-reach drainage area and
VBET class, which is useful for stratifying training data or reporting results by stream size.